In [1]:
# Layer 5 — Data Quality Framework
# Pipeline: LaDe Delivery Intelligence
# Script: 05_quality_check.ipynb
# Purpose: Run automated quality checks on star schema

import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
from datetime import datetime

CONFIG = {
    "db_host": "localhost",
    "db_user": "root",
    "db_password": "password",
    "db_name": "lade_pipeline"
}

password = urllib.parse.quote_plus(CONFIG['db_password'])
engine = create_engine(
    f"mysql+mysqlconnector://{CONFIG['db_user']}:{password}@{CONFIG['db_host']}/{CONFIG['db_name']}"
)

# Results storage
quality_results = []

print("Data Quality Framework started")
print(f"Run time: {datetime.now()}")

Data Quality Framework started
Run time: 2026-05-06 22:10:45.258578


In [7]:
# quality checks

def log_check(check_name, status, actual_value, expected, message):
    quality_results.append({
        'check_name': check_name,
        'status': status,
        'actual_value': str(actual_value),
        'expected': str(expected),
        'message': message,
        'checked_at': datetime.now()
    })
    print(f"{'PASS' if status == 'PASS' else ' FAIL'} | {check_name} | {message}")

# Check 1 — Row count
actual_rows = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_deliveries", engine).iloc[0]['cnt']
expected_rows = 472419
status = 'PASS' if actual_rows == expected_rows else 'FAIL'
log_check('row_count', status, actual_rows, expected_rows, f"Expected {expected_rows}, got {actual_rows}")

# Check 2 — Valid record rate
valid_count = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_deliveries WHERE is_valid = 1", engine).iloc[0]['cnt']
valid_pct = round(valid_count / actual_rows * 100, 2)
status = 'PASS' if valid_pct >= 95 else 'FAIL'
log_check('valid_rate', status, valid_pct, '>=95%', f"Valid records: {valid_pct}%")

# Check 3 — Null check on key columns
null_count = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_deliveries WHERE city_id IS NULL OR courier_id IS NULL OR date_id IS NULL", engine).iloc[0]['cnt']
status = 'PASS' if null_count == 0 else 'FAIL'
log_check('null_foreign_keys', status, null_count, 0, f"Null foreign keys: {null_count}")

# Check 4 — Duplicate order_ids
dup_count = pd.read_sql("SELECT COUNT(*) - COUNT(DISTINCT order_id) as cnt FROM fact_deliveries", engine).iloc[0]['cnt']
status = 'PASS' if dup_count == 0 else 'FAIL'
log_check('duplicate_orders', status, dup_count, 0, f"Duplicate order_ids: {dup_count}")

# Check 5 — Duration range
invalid_duration = pd.read_sql("SELECT COUNT(*) as cnt FROM fact_deliveries WHERE delivery_duration_minutes < 0 AND is_valid = 1", engine).iloc[0]['cnt']
status = 'PASS' if invalid_duration == 0 else 'FAIL'
log_check('duration_range', status, invalid_duration, 0, f"Invalid durations in valid records: {invalid_duration}")

# Check 6 — City coverage
city_count = pd.read_sql("SELECT COUNT(DISTINCT city_id) as cnt FROM fact_deliveries", engine).iloc[0]['cnt']
status = 'PASS' if city_count == 3 else 'FAIL'
log_check('city_coverage', status, city_count, 3, f"Cities present: {city_count} of 3 expected")

# Check 7 — All dimension tables populated
dim_checks = {
    'dim_city': 3,
    'dim_courier': 1870,
    'dim_date': 15,
    'dim_depot': 134
}
for table, expected in dim_checks.items():
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", engine).iloc[0]['cnt']
    status = 'PASS' if count == expected else 'FAIL'
    log_check(f'dim_{table}_count', status, count, expected, f"{table}: {count} rows")

print(f"\n{'='*50}")
print(f"Quality checks complete: {sum(1 for r in quality_results if r['status'] == 'PASS')} PASS, {sum(1 for r in quality_results if r['status'] == 'FAIL')} FAIL")

PASS | row_count | Expected 472419, got 472419
PASS | valid_rate | Valid records: 98.51%
PASS | null_foreign_keys | Null foreign keys: 0
PASS | duplicate_orders | Duplicate order_ids: 0
PASS | duration_range | Invalid durations in valid records: 0
PASS | city_coverage | Cities present: 3 of 3 expected
PASS | dim_dim_city_count | dim_city: 3 rows
PASS | dim_dim_courier_count | dim_courier: 1870 rows
PASS | dim_dim_date_count | dim_date: 15 rows
PASS | dim_dim_depot_count | dim_depot: 134 rows

Quality checks complete: 20 PASS, 0 FAIL


In [9]:
# Saving quality results to database
quality_df = pd.DataFrame(quality_results)
quality_df.to_sql('data_quality_log', engine, if_exists='append', index=False)
print("Quality results saved to data_quality_log table")
print(f"Total checks logged: {len(quality_df)}")

Quality results saved to data_quality_log table
Total checks logged: 20
